# try-except-solve — ex1: graceful single-call solve with try/except

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `try-except-solve`. Running the final beacon cell reports progress against the `LinAlg: try/except solve` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `LinAlg: try/except solve` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`try-except-solve`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "try-except-solve"
DD_SUBTOPIC = "LinAlg: try/except solve"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `try`/`except` around `torch.linalg.solve` — quick refresher

`torch.linalg.solve(A, b)` raises `torch._C._LinAlgError` (a subclass of `RuntimeError`) when `A` is singular. In a hot batch this means ONE bad matrix kills the whole solve.

**Two strategies for singular A:**
1. **`try`/`except` fallback to None.** Wrap the call; on error return a sentinel (`None`, `nan`, a flag). Caller decides what to do. Simple, honest, exposes the failure.
2. **Identity-substitution trick.** Replace singular A's with I before solving (different atom: `singular-matrix-mask-trick`). Lets you keep the tensor shape and mask out failures downstream.

**This drill drills strategy 1.** Use the try/except pattern for non-batched solves where graceful per-call failure is acceptable.

### Exercise 1 — graceful single-call solve with try/except

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `try`/`except RuntimeError` around `torch.linalg.solve` so a singular matrix returns `None` instead of crashing.
> Keywords: try-except, linalg-solve, singular, graceful-failure, fallback
> ```

**KCs targeted:** `catch-linalg-runtime-error`, `return-none-on-singular`

Implement `ex1_safe_solve(A, b)`:

1. Try `t.linalg.solve(A, b)`. If it succeeds, return the result.
2. If it raises `RuntimeError` (covers `_LinAlgError`), return `None`.

Signature: `(A: Tensor, b: Tensor) -> Optional[Tensor]`.

`A` is `(n, n)`, `b` is `(n,)` for this drill. Returns either the solution `(n,)` or `None`.

**Contrast** with `singular-matrix-mask-trick`: that atom would substitute identity and return a tensor of shape `(n,)` with `nan` / zero markers. This atom is for the non-batched case where you want a clean Python None and the caller branches on it.

In [ ]:
def ex1_safe_solve(A: Tensor, b: Tensor) -> Optional[Tensor]:
    try:
        return t.linalg.solve(A, b)
    except RuntimeError:
        return None


<details><summary>Solution</summary>

```python
def ex1_safe_solve(A: Tensor, b: Tensor) -> Optional[Tensor]:
    try:
        return t.linalg.solve(A, b)
    except RuntimeError:
        return None
```

**Why `RuntimeError` not `_LinAlgError`.** `torch._C._LinAlgError` is a SUBCLASS of `RuntimeError`. Catching the parent is portable across PyTorch versions (the linalg error type was renamed in 2.0+) AND across CUDA-vs-CPU codepaths (CUDA sometimes raises raw `RuntimeError` with a different message).

**Why None not nan.** A `None` return is a Python sentinel that forces the caller to branch. A `nan` tensor silently propagates and can poison downstream computation — usually the OPPOSITE of what you want for a graceful-failure abstraction.

**When NOT to use this.** In a hot batched loop (1000s of solves), the try/except overhead is per-call and Python-side. For that case use the masked-substitution trick (`singular-matrix-mask-trick`) which stays vectorized.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()